# Camada Bronze - Ingestão

## Objetivo
Ingerir os 5 arquivos CSV da Landing Zone e a cotação do dólar obtida via API do BCB,
preservando os dados brutos e adicionando `ingestion_datetime` em cada registro.


### Padronização dos nomes
Embora no enunciado fale para dar omitir o catalogo do nome da tabela eu optei por incluir como mostrado em aula para ficar um nome mais robusto.


In [0]:
catalog = "movies_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: movies_medallion
bronze_schema: movies_medallion.bronze
silver_schema: movies_medallion.silver
gold_schema: movies_medallion.gold
landing_path: /Volumes/movies_medallion/bronze/Landing


### Ingestão dos dados

Sei que no enunciado fala pra usar `.mode("append")` mas como as tabelas são adicionadas em uma única batelada optei por usar `.mode("overwrite")` para evitar adicionar dados repetidos ao rodar o notebook mais de uma vez. 

In [0]:
from pyspark.sql.functions import current_timestamp

expected_files = ['movies_info_IMDB_TMDB.csv',
                  'movies_financials_IMDB_TMDB.csv',
                  'movies_metrics_IMDB_TMDB.csv',
                  'credits_and_tags_IMDB_TMDB.csv',
                  'movies_reviews.csv']

# Padroniza o nome da tabela a partir do arquivo: remove o sufixo do fornecedor e a extensão ".csv"
for file in expected_files:
    name = file.replace("_IMDB_TMDB", "").removesuffix(".csv")

    # Adiciona o timestamp de quando a tabela foi inserida.
    df = (spark.read.csv(f"{landing_path}/{file}", header=True, inferSchema=True)
            .withColumn("ingestion_datetime", current_timestamp()))
    
    # Define o formato Delta e adiciona o prefixo .tb_.
    df.write.format("delta").mode("overwrite").saveAsTable(f"{bronze_schema}.tb_{name}")

    # Mostra brevemente os resultados.
    print(f"{bronze_schema}.tb_{name}")
    display(spark.table(f"{bronze_schema}.tb_{name}").limit(5))
  

movies_medallion.bronze.tb_movies_info


id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-10-07T21:41:23.480Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-10-07T21:41:23.480Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-10-07T21:41:23.480Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-10-07T21:41:23.480Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-10-07T21:41:23.480Z


movies_medallion.bronze.tb_movies_financials


id,budget,revenue,ingestion_datetime
293660,58000000,Unknown,2026-10-07T21:41:28.068Z
299536,300000000,2052415039,2026-10-07T21:41:28.068Z
299534,356000000,2800000000,2026-10-07T21:41:28.068Z
475557,55000000,1074458282,2026-10-07T21:41:28.068Z
271110,250000000,Não Informado,2026-10-07T21:41:28.068Z


movies_medallion.bronze.tb_movies_metrics


id,popularity,vote_average,vote_count,averageRating,numVotes,ingestion_datetime
293660,72.735,7.606,28894,8.0,1270339,2026-10-07T21:41:31.763Z
299536,"154,34",8.255,27713,8.4,1406782,2026-10-07T21:41:31.763Z
299534,91.756,8.263,23857,8.4,1484150,2026-10-07T21:41:31.763Z
475557,"54,522",8.168,23425,8.3,1723035,2026-10-07T21:41:31.763Z
271110,70.741,7.4,21541,7.8,947222,2026-10-07T21:41:31.763Z


movies_medallion.bronze.tb_credits_and_tags


id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-07T21:41:35.556Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-07T21:41:35.556Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-07T21:41:35.556Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-07T21:41:35.556Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-07T21:41:35.556Z


movies_medallion.bronze.tb_movies_reviews


id,nome,nota,comentario,ingestion_datetime
442113,Mariana Cardoso 277,4.4,null,2026-10-07T21:41:39.682Z
637007,Lucas Reis 602,3.9,null,2026-10-07T21:41:39.682Z
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.,2026-10-07T21:41:39.682Z
413036,Gabriela Monteiro 401,7.5,null,2026-10-07T21:41:39.682Z
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.,2026-10-07T21:41:39.682Z


## Ingestão dados API

Não consigo acessar internet pelo databricks (`NameResolutionError` em `olinda.bcb.gov.br`).
Sendo assim eu obtive o JSON fora do databricks e fiz a ingestão dele.

- Endpoint: `CotacaoDolarPeriodo` da API PTAX/BCB
- Período: 29/09/2026 a 06/10/2026
- Retorno: 6 cotações (sábados e domingos sem cotação, conforme esperado)
- Arquivo salvo: `cotacao_dolar.json` e enviado para a landing zone

O subsequente tratamento do JSON foi feito totalmente no databricks seguindo os padrões definidos.


In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp, current_timestamp

# Caminho do arquivo JSON obtido da API do BCB
json_path = f"{landing_path}/cotacao_dolar.json"

# Lê o JSON 
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Retorna os dados
cotacao_dolar = data.get("value", [])
print(f"Cotações carregadas: {len(cotacao_dolar)}")
print(f"Exemplo: {cotacao_dolar[0]}")

Cotações carregadas: 6
Exemplo: {'cotacaoCompra': 5.2198, 'dataHoraCotacao': '2026-09-29 13:06:06.446305'}


In [0]:
schema = StructType([
    StructField("cotacaoCompra",   DoubleType(), True),
    StructField("dataHoraCotacao", StringType(), True),
])

df = spark.createDataFrame(cotacao_dolar, schema=schema)

# Converte dataHoraCotacao (string) para timestamp e adiciona o carimbo do Bronze
df = (df
    .withColumn("dataHoraCotacao", to_timestamp("dataHoraCotacao"))
    .withColumn("ingestion_datetime", current_timestamp()))

df.printSchema()
df.show(truncate=False)

root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: timestamp (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = false)

+-------------+--------------------------+--------------------------+
|cotacaoCompra|dataHoraCotacao           |ingestion_datetime        |
+-------------+--------------------------+--------------------------+
|5.2198       |2026-09-29 13:06:06.446305|2026-10-07 21:41:42.832974|
|5.1803       |2026-09-30 13:11:44.783262|2026-10-07 21:41:42.832974|
|5.2073       |2026-10-01 13:10:35.4469  |2026-10-07 21:41:42.832974|
|5.2232       |2026-10-02 13:03:16.256632|2026-10-07 21:41:42.832974|
|4.9853       |2026-10-05 13:07:36.558602|2026-10-07 21:41:42.832974|
|4.9692       |2026-10-06 13:03:21.267287|2026-10-07 21:41:42.832974|
+-------------+--------------------------+--------------------------+



In [0]:
table_name = f"{bronze_schema}.tb_cotacao_dolar"

(df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name))

print(f"✅ Gravado em: {table_name}")
display(spark.table(table_name).orderBy("dataHoraCotacao"))

✅ Gravado em: movies_medallion.bronze.tb_cotacao_dolar


cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.2198,2026-09-29T13:06:06.446Z,2026-10-07T21:41:43.889Z
5.1803,2026-09-30T13:11:44.783Z,2026-10-07T21:41:43.889Z
5.2073,2026-10-01T13:10:35.446Z,2026-10-07T21:41:43.889Z
5.2232,2026-10-02T13:03:16.256Z,2026-10-07T21:41:43.889Z
4.9853,2026-10-05T13:07:36.558Z,2026-10-07T21:41:43.889Z
4.9692,2026-10-06T13:03:21.267Z,2026-10-07T21:41:43.889Z


In [0]:
df_check = spark.table(f"{bronze_schema}.tb_cotacao_dolar")
print(f"Linhas: {df_check.count()}")   # espera 6
df_check.orderBy("dataHoraCotacao").show(truncate=False)

Linhas: 6
+-------------+--------------------------+--------------------------+
|cotacaoCompra|dataHoraCotacao           |ingestion_datetime        |
+-------------+--------------------------+--------------------------+
|5.2198       |2026-09-29 13:06:06.446305|2026-10-07 21:41:43.889078|
|5.1803       |2026-09-30 13:11:44.783262|2026-10-07 21:41:43.889078|
|5.2073       |2026-10-01 13:10:35.4469  |2026-10-07 21:41:43.889078|
|5.2232       |2026-10-02 13:03:16.256632|2026-10-07 21:41:43.889078|
|4.9853       |2026-10-05 13:07:36.558602|2026-10-07 21:41:43.889078|
|4.9692       |2026-10-06 13:03:21.267287|2026-10-07 21:41:43.889078|
+-------------+--------------------------+--------------------------+

